# FinTok · Pipeline de texto modular

Este notebook prueba el pipeline final utilizando los módulos de `src/text/`.

Arquitectura:

`chunk_selector → fact_extractor → fact_checker → script_generator → JSON final`

La prueba usa un único ejemplo (`AAPL-2024-8-0043`) para comprobar el pipeline completo antes de automatizarlo sobre todo el corpus.


In [1]:
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes huggingface_hub pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 25.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


## 1. Clonar FinTok y cargar el corpus

El notebook asume que los cuatro módulos `.py` están guardados en `src/text/` dentro del repositorio.


In [11]:
from pathlib import Path
import sys
import json
import time
import zipfile
import pandas as pd
import torch

if not Path("/content/FinTok").exists():
    !git clone https://github.com/mirdbg/FinTok.git

REPO_DIR = Path("/content/FinTok")
ZIP_PATH = REPO_DIR / "data" / "corpus_miax_2026.zip"
WORK_DIR = Path("/content/corpus_miax_2026")

WORK_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"No encuentro: {ZIP_PATH}"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(WORK_DIR)

with open(WORK_DIR / "chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(chunks)

# Permitir imports desde /content/FinTok
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Chunks cargados: {len(df):,}")
display(df[["chunk_id", "ticker", "fiscal_year", "item", "posicion"]].head())


Chunks cargados: 1,749


,chunk_id,ticker,fiscal_year,item,posicion
0,NVDA-2024-1A-0000,NVDA,2024,1A,0
1,NVDA-2024-1A-0001,NVDA,2024,1A,1
2,NVDA-2024-1A-0002,NVDA,2024,1A,2
3,NVDA-2024-1A-0003,NVDA,2024,1A,3
4,NVDA-2024-1A-0004,NVDA,2024,1A,4


## 2. Cargar Gemma 3 4B IT

El modelo se carga una sola vez en el notebook. Los módulos reciben después `ask_gemma` como función de generación, evitando que cada componente cargue su propia copia del modelo.


In [3]:
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

MODEL_ID = "google/gemma-3-4b-it"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=quant_config,
)

model.eval()

print("Modelo cargado:", MODEL_ID)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Modelo cargado: google/gemma-3-4b-it


In [4]:
def ask_gemma(system_prompt, user_prompt, max_new_tokens=600):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start
    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return text, elapsed


## 3. Importar los componentes del pipeline

Cada módulo tiene una responsabilidad única. El notebook solo orquesta las llamadas.


In [12]:
from src.text.chunk_selector import get_context
from src.text.fact_extractor import extract_facts
from src.text.fact_checker import select_fact
from src.text.script_generator import generate_script, build_video_json


## 4. Elegir un ejemplo

Usamos como chunk de referencia `AAPL-2024-8-0043`. `get_context()` recupera automáticamente tres chunks consecutivos de la misma sección.


In [6]:
CHUNK_ID = "GOOGL-2024-8-0037"

context_df = get_context(
    df,
    chunk_id=CHUNK_ID
)

display(
    context_df[
        ["chunk_id", "ticker", "fiscal_year", "item", "posicion"]
    ]
)


,chunk_id,ticker,fiscal_year,item,posicion
36,GOOGL-2024-8-0036,GOOGL,2024,8,36
37,GOOGL-2024-8-0037,GOOGL,2024,8,37
38,GOOGL-2024-8-0038,GOOGL,2024,8,38


## 5. Paso 1 · EXTRACT

Se extraen hasta tres ideas factuales, manteniendo evidencia y procedencia.


In [20]:
extraction = extract_facts(
    context_df=context_df,
    generate_fn=ask_gemma
)

print(json.dumps(
    extraction["data"],
    indent=2,
    ensure_ascii=False
))


{
  "ideas": [
    {
      "idea": "Unrealized losses increased significantly for government bonds, corporate debt securities, and mortgage-backed securities as of December 31, 2024, totaling $214 million.",
      "evidence": "As of December 31, 2024\nLess than 12 Months\nFair Value\nGovernment bonds $11,119\nCorporate debt securities ( 126 )\nMortgage-backed and asset-backed securities ( 385 )",
      "source_chunks": [
        "CHUNK GOOGL-2024-8-0036"
      ]
    },
    {
      "idea": "In 2024, the total gain on equity securities in OI&E was $3,714 million, primarily driven by realized gains on marketable and non-marketable equity securities.",
      "evidence": "Year Ended December 31, 2024\nTotal gain (loss) on equity securities in other income (expense), net $3,714",
      "source_chunks": [
        "CHUNK GOOGL-2024-8-0038"
      ]
    },
    {
      "idea": "The carrying value of non-marketable equity securities increased substantially from $21,240 million to $35,531 million a

## 6. Paso 2 · SELECT / FACT CHECK

El editor decide si alguna idea es suficientemente relevante para FinTok. Si devuelve `DISCARD`, el pipeline debe detenerse y no generar un guion.


In [21]:
selection = select_fact(
    context_df=context_df,
    extraction=extraction,
    generate_fn=ask_gemma
)

print(json.dumps(
    selection["data"],
    indent=2,
    ensure_ascii=False
))


{
  "decision": "KEEP",
  "selected_idea": "The carrying value of non-marketable equity securities increased substantially from $21,240 million to $35,531 million as of December 31, 2023 and 2024, respectively.",
  "reason": "This represents a significant shift in the company's equity portfolio and provides insight into the company's investment strategy and potential valuation changes. The substantial increase in carrying value, coupled with the details about cumulative net gains and losses, offers a clear and understandable picture of how the company's non-marketable equity investments are performing. It's a key metric that investors would want to track and understand, and the provided evidence directly supports this claim."
}


## 7. Paso 3 · WRITE

Solo si la decisión es `KEEP`, Gemma genera el hook, la explicación y los dos highlights visuales.


In [22]:
if selection["data"]["decision"].upper() == "KEEP":
    script = generate_script(
        context_df=context_df,
        selection=selection,
        generate_fn=ask_gemma
    )

    print(json.dumps(
        script["data"],
        indent=2,
        ensure_ascii=False
    ))
else:
    script = None
    print("DISCARD: no se genera script.")


{
  "hook": "Did you know Alphabet's equity investments jumped dramatically in 2024? What's driving this shift?",
  "hook_highlight": "Equity Investment Increase",
  "explanation": "Alphabet’s non-marketable equity securities jumped from $21.2 billion to $35.5 billion between 2023 and 2024. This increase is primarily due to unrealized gains, particularly in government bonds and corporate debt securities. These gains reflect changes in market value, and are recognized as a component of OI&E.  The company’s strategy and valuation are being impacted by these shifts.",
  "explanation_highlight": "Increase in Non-Marketable Equity Securities"
}


## 8. Paso 4 · JSON final

Esta fase no utiliza IA. Python añade la intro y el outro fijos y construye las cuatro escenas con la estructura que consumirán después las fases de audio y vídeo.


In [23]:
if script is not None:
    video_json = build_video_json(
        context_df=context_df,
        script=script
    )

    print(json.dumps(
        video_json,
        indent=2,
        ensure_ascii=False
    ))
else:
    video_json = None
    print("No hay vídeo porque el contexto fue descartado.")


{
  "company": "GOOGL",
  "year": 2024,
  "language": "en",
  "scenes": [
    {
      "scene_id": 1,
      "scene_type": "hook",
      "speaker": null,
      "text": "Did you know Alphabet's equity investments jumped dramatically in 2024? What's driving this shift?",
      "key_figure": "Equity Investment Increase",
      "source_chunk_ids": [
        "GOOGL-2024-8-0036",
        "GOOGL-2024-8-0037",
        "GOOGL-2024-8-0038"
      ],
      "verified": true,
      "audio_path": null
    },
    {
      "scene_id": 2,
      "scene_type": "intro",
      "speaker": null,
      "text": "This is FinTok — your daily dose of financial knowledge.",
      "key_figure": null,
      "source_chunk_ids": [],
      "verified": true,
      "audio_path": null
    },
    {
      "scene_id": 3,
      "scene_type": "explanation",
      "speaker": null,
      "text": "Alphabet’s non-marketable equity securities jumped from $21.2 billion to $35.5 billion between 2023 and 2024. This increase is primarily d

## 9. Pipeline completo en una función

Una vez comprobadas las etapas individualmente, esta función permite ejecutar exactamente el mismo flujo con cualquier `chunk_id`.


In [24]:
def run_example(chunk_id):
    context = get_context(df, chunk_id)

    extraction = extract_facts(
        context_df=context,
        generate_fn=ask_gemma
    )

    selection = select_fact(
        context_df=context,
        extraction=extraction,
        generate_fn=ask_gemma
    )

    if selection["data"]["decision"].upper() != "KEEP":
        return {
            "status": "DISCARD",
            "chunk_id": chunk_id,
            "selection": selection["data"]
        }

    script = generate_script(
        context_df=context,
        selection=selection,
        generate_fn=ask_gemma
    )

    return build_video_json(
        context_df=context,
        script=script
    )


### Ejecución end-to-end opcional

Esta celda vuelve a ejecutar las tres inferencias. No hace falta lanzarla si ya has probado las etapas 5–8; está preparada para comprobar el uso final de la API modular.


In [ ]:
# Descomenta cuando quieras probar la función end-to-end:
#
# result = run_example("AAPL-2024-8-0043")
# print(json.dumps(result, indent=2, ensure_ascii=False))
